In [1]:
%pip install -Uq langchain langchain-core dotenv


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

parser = StrOutputParser()

chain_1 = (
    ChatPromptTemplate.from_template("{topic}을 한 문장으로 요약해줘")
    | llm 
    | parser
)

chain_2 = (
    ChatPromptTemplate.from_template("{topic}의 실제 사용 예시를 한 가지만 알려줘.")
    | llm 
    | parser
)

parallel = RunnableParallel(
    chain_1=chain_1,
    chain_2=chain_2
)

result = parallel.invoke({
    'topic': 'python'
})

print(type(result))
print(result.keys())

print()

print('첫번째 결과')
print(result['chain_1'])
print()

print('두번째 결과')
print(result['chain_2'])

# ( "human", "파이썬이 뭔지 알려줘")

/Users/woocheolseong/work/knu_ai_rag/venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


<class 'dict'>
dict_keys(['chain_1', 'chain_2'])

첫번째 결과
Python은 빠르게 개발할 수 있는 고수준의 프로그래밍 언어로, 데이터 분석, 웹 개발, 인공지능 및 머신 러닝 등 다양한 응용 프로그램에 사용되는 멀티 패러다임의 프로그래밍 언어입니다.

두번째 결과
Python은 다양한 분야에서 사용되는 유용한 언어입니다. 실제 사용 예시 중 하나는 **웹 스크래핑**입니다.

웹 스크래핑은 특정 웹사이트의 데이터를 추출하여 분석하거나 저장하는 기술입니다. Python의 **BeautifulSoup** 라이브러리와 **requests** 라이브러리와 같은 도구를 사용하여 쉽게 웹 페이지를 파싱하고 데이터를 추출할 수 있습니다.

예를 들어, 다음은 Python을 사용하여 네이버 뉴스 페이지에서 뉴스 제목과 링크를 추출하는 예시입니다.

```python
import requests
from bs4 import BeautifulSoup

# 네이버 뉴스 페이지 URL
url = "https://news.naver.com/main/read.nhn?oid=001&aid=0000012160"

# 요청 헤더
headers = {'User-Agent': 'Mozilla/5.0'}

# 요청 보내기
response = requests.get(url, headers=headers)

# HTML 파싱
soup = BeautifulSoup(response.text, 'html.parser')

# 뉴스 제목과 링크 추출
news_titles = soup.find_all('span', class_='title')
news_links = [a.get('href') for a in soup.find_all('a')]

# 출력
for title, link in zip(news_titles, news_links):
    print(title.text, link)
```

이 코드는 네이버 뉴스 페이지에서 뉴스 제목과 링크를 추출하여 출력합니

In [ ]:
from langchain_core.runnables import RunnableLambda

pros_chain = ChatPromptTemplate.from_template("{topic}의 장점 2가지") | llm | parser
cons_chain = ChatPromptTemplate.from_template("{topic}의 단점 2가지") | llm | parser

parallel_1 = RunnableParallel(
    pros=pros_chain,
    cons=cons_chain
)

parallel_2 = {
    "pros": pros_chain,
    "cons": cons_chain
}

chain = parallel_2 | RunnableLambda(lambda x: print(x))

result = chain.invoke({
    'topic': 'Java'
})

print(result)


{'pros': 'Java의 2가지 장점은 다음과 같습니다.\n\n1. **다중 플랫폼 지원**: Java는 플랫폼 독립적인 언어로, 하나의 Java 코드를 여러 운영체제(Windows, macOS, Linux 등)에 배포할 수 있습니다. Java의 자바 가상 머신(JVM)으로 인해 Java 코드는 운영체제에 독립적이기 때문에 다양한 운영체제에서 실행할 수 있습니다.\n\n2. **객체 지향 프로그래밍**: Java는 객체 지향 프로그래밍(OOP) 언어로, 클래스와 객체를 통해 프로그램을 개발할 수 있습니다. 객체 지향 프로그래밍은 프로그램을 작은 단위의 객체로 구분하여 개발하는 방식으로, 코드 재사용이 facile합니다. Java의 객체 지향 프로그래밍 기능은 프로그램 개발을 간편하고 유지보수가 용이하게 합니다.', 'cons': 'Java의 단점은 다음과 같습니다.\n\n1. **가독성**: Java의 코드는 종종 복잡하고 장황한 코드를 포함할 수 있습니다. Java는 객체지향 프로그래밍(OOP) 개념을 사용하므로 클래스와 객체를 정의하는 데 많은 코드가 필요할 수 있습니다. 이러한 코드는 가독성이 떨어질 수 있으며, 다른 프로그래머들이 코드를 읽고 이해하기 어려울 수 있습니다.\n\n2. **성능**: Java는 interpreted 언어이므로, 실행에 필요한 자원과 시간이 더 많이 필요합니다. Java의 가비지 컬렉터(Garbage Collector)가 메모리 관리를 자동으로 수행한다는 장점이 있지만, 가비지 컬렉션의 과정에 시간이 소요되며, 이 때문에 Java의 성능이 다른 언어와 비교하여 떨어질 수 있습니다.'}
None


In [4]:
inspect_a = RunnableLambda(lambda x: f"브랜치 A가 받은 입력: {x}")
inspect_b = RunnableLambda(lambda x: f"브랜치 B가 받은 입력: {x}")

parallel = RunnableParallel(
    a = inspect_a,
    b = inspect_b
)

result = parallel.invoke(
    {
        'name': '대전',
        'location': '충청남도'
    }
)

print(f'a -> {result['a']}')
print(f'b -> {result['b']}')




a -> 브랜치 A가 받은 입력: {'name': '대전', 'location': '충청남도'}
b -> 브랜치 B가 받은 입력: {'name': '대전', 'location': '충청남도'}


In [6]:

extract_question = RunnableLambda(lambda x: f"질문 : {x["question"]}")
extract_context_len = RunnableLambda(lambda x: f"컨텍스트 길이: {len(x['context'])}")
extract_language = RunnableLambda(lambda x: f"질문 작성 언어: {x['lang']}")

parallel = RunnableParallel(
    question = extract_question,
    context = extract_context_len,
    lang = extract_language
)

result = parallel.invoke({
    'question': '머신러닝은 무엇인가요?',
    'context': '머신러닝은 데이넡에서 패턴을 학습하는 인공지능 기법입니다',
    'lang': 'Korean'
})

print(f'question -> {result['question']}')
print(f'context -> {result['context']}')
print(f'lang -> {result['lang']}')





question -> 질문 : 머신러닝은 무엇인가요?
context -> 컨텍스트 길이: 31
lang -> 질문 작성 언어: Korean


In [ ]:
parallel = RunnableParallel(
    question=RunnableLambda(lambda x: x["question"]),
    lang=RunnableLambda(lambda x: x['lang'])
)

merge_fn = RunnableLambda(
    lambda x: {
        "question": x["question"],
        "lang": x["lang"],
    }
)

answer_prompt = ChatPromptTemplate.from_template(
    "{question}에 대해서 {lang}로 두 문장 이내로 답변해줘"
)

chain = parallel | merge_fn | answer_prompt | llm | parser

answer = chain.invoke({
    'question': '인공지능이 무엇인지 설명해줘',
    'lang': '중국어',
})


print(answer)


인공지능( AI )은 컴퓨터 프로그램을 통해 인간의 지능을 모방하여 해결하는 기술입니다. 인공지능은 데이터를 학습하고 학습한 데이터를 바탕으로 문제를 해결하고 미래를 예측하는 기능을 가집니다.


In [10]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough.assign(
    length = RunnableLambda(lambda x: len(x['text'])),
    upper = RunnableLambda(lambda x: x['text'].upper()),
)

result = passthrough.invoke({
    'text': 'hello world!'
})

print(result)


{'text': 'hello world!', 'length': 12, 'upper': 'HELLO WORLD!'}


In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

chain = RunnableParallel(
    input = RunnablePassthrough(),
    process = RunnableLambda(lambda x: x['text'].upper())
)

result = chain.invoke({
    'text': 'hello, world!'
})

print(result)



{'input': {'text': 'hello, world!'}, 'process': 'HELLO, WORLD!'}
